# NYC Collision Severity: Feature Store & Feature Group Setup
This notebook demonstrates how to store engineered features in the **SageMaker Feature Store**, enabling both real-time serving (online) and offline model training.

In [ ]:
!pip install "sagemaker<3.0.0"

import boto3
import sagemaker
from sagemaker.session import Session
from sagemaker.feature_store.feature_group import FeatureGroup
import pandas as pd
import numpy as np
import time
import sys
import os
from time import gmtime, strftime

# Add src to path for function imports
sys.path.append('../src')
from preprocessing import preprocess_pipeline
from features import feature_engineering_pipeline

# 1. Setup Sessions and Clients
region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

default_bucket = feature_store_session.default_bucket()
role = sagemaker.get_execution_role()
print(f"Using Role: {role}")

### Load and Engineer Features
We use the core scripts from `src/` to ensure consistency.

In [ ]:
# Load the data
df = pd.read_csv('../data/raw/merged_sample.csv')

# Apply Pipeline
df = preprocess_pipeline(df)
df = feature_engineering_pipeline(df)

# Add Feature Store mandatory columns
# 1. EventTime
df['EventTime'] = time.time()

# 2. Record Identifier (Ensure collision_id is string and unique)
df['collision_id'] = df['collision_id'].astype(str)

# Select a subset of features for the group
feature_set = [
    'collision_id', 'EventTime', 'target', 
    'month', 'crash_hour', 'rush_hour', 'is_weekend', 
    'factor_category', 'vehicle_type', 'borough'
]
feature_df = df[feature_set].copy()

def cast_object_to_string(data_frame):
    for label in data_frame.columns:
        if data_frame.dtypes[label] == "object" or data_frame.dtypes[label] == "string":
            data_frame[label] = data_frame[label].astype("str").astype("string")

cast_object_to_string(feature_df)
print(f"Feature set ready with shape: {feature_df.shape}")

### Define and Create Feature Group

In [ ]:
feature_group_name = "nyc-collision-group-" + strftime("%d-%H-%M-%S", gmtime())

collision_feature_group = FeatureGroup(
    name=feature_group_name, sagemaker_session=feature_store_session
)

collision_feature_group.load_feature_definitions(data_frame=feature_df)

collision_feature_group.create(
    s3_uri=f"s3://{default_bucket}/aai-540-group6/feature-store/",
    record_identifier_name="collision_id",
    event_time_feature_name="EventTime",
    role_arn=role,
    enable_online_store=True
)

def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation...")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")

wait_for_feature_group_creation_complete(collision_feature_group)

### Ingest Records

In [ ]:
collision_feature_group.ingest(data_frame=feature_df, max_workers=3, wait=True)
print("Ingestion complete.")

### Query Feature Store (Online Store)

In [ ]:
def query_collision(cid):
    print(f"*** Querying Collision ID: {cid} ***")
    response = featurestore_runtime.get_record(
        FeatureGroupName=feature_group_name,
        RecordIdentifierValueAsString=str(cid)
    )
    if "Record" in response:
        return pd.DataFrame(response["Record"])
    else:
        return "Record not found."

# Sample Query
sample_cid = feature_df.iloc[0]['collision_id']
query_collision(sample_cid)